# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a demonstration of loading, exploring, and analyzing a Croissant-data-standard dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant schema URL:** [`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)
- **Dataset DOI:** `10.71728/senscience.vcs2-05nj`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list the available record sets, their fields, and field `@id`s for reference.


In [ ]:
# Discover available record sets and their fields
print('Available record sets and their fields:')
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"\nRecord set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}) [type={field.data_type}]")

## 3. Data Extraction
Load data from each record set into a DataFrame using their `@id`.

Let's extract all top-level tabular record sets into pandas DataFrames for analysis.

In [ ]:
# Create a dictionary of DataFrames for record sets
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded record set: {rs.name} (@id={rs.id}) with shape {df.shape}")
    else:
        print(f"No records found for record set: {rs.name} (@id={rs.id})")

# Print the DataFrame columns for each loaded record set
print('\nAvailable DataFrames and columns:')
for rsid, df in dataframes.items():
    print(f"Record set @id: {rsid}")
    print("  Columns:", df.columns.tolist())
    print("  Preview:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps for one main record set. We will:
- Filter rows based on a numeric field (e.g., 'age')
- Normalize a numeric field (e.g., 'phq9_score')
- Group and summarize by a categorical field (e.g., 'gender')

**Note:** Please check from above the actual field `@id`s used in your dataset and update the variables below accordingly.

In [ ]:
# Example: Select the main survey record set and fields by their @ids

# Set these to the correct @ids from section 2 output above (example shown, adjust if needed!)
RECORD_SET_ID = None
AGE_FIELD_ID = None
SCORE_FIELD_ID = None
GROUP_FIELD_ID = None

# Try to match main tabular record set
for rsid, df in dataframes.items():
    columns = df.columns.tolist()
    # Guess if obvious fields exist
    age_id = None
    score_id = None
    group_id = None
    for col in columns:
        if 'age' in col.lower() and age_id is None:
            age_id = col
        if 'phq' in col.lower() and score_id is None:
            score_id = col
        if 'gender' in col.lower() and group_id is None:
            group_id = col
    if age_id and score_id and group_id:
        RECORD_SET_ID = rsid
        AGE_FIELD_ID = age_id
        SCORE_FIELD_ID = score_id
        GROUP_FIELD_ID = group_id
        break

if RECORD_SET_ID is None:
    raise Exception("Could not determine record set and field IDs automatically. Please set RECORD_SET_ID, AGE_FIELD_ID, SCORE_FIELD_ID, GROUP_FIELD_ID based on section 2 output.")

df = dataframes[RECORD_SET_ID]
print(f"Using record set @id: {RECORD_SET_ID}, age field: {AGE_FIELD_ID}, score field: {SCORE_FIELD_ID}, group field: {GROUP_FIELD_ID}")

# 1. Filtering: Only participants age > 18
filtered_df = df[df[AGE_FIELD_ID] > 18]
print(f"Number of records with {AGE_FIELD_ID} > 18: {len(filtered_df)}")
display(filtered_df[[AGE_FIELD_ID, SCORE_FIELD_ID, GROUP_FIELD_ID]].head())

# 2. Normalization: Z-score for PHQ-9 score field
mean_score = filtered_df[SCORE_FIELD_ID].mean()
std_score = filtered_df[SCORE_FIELD_ID].std()
filtered_df[f"{SCORE_FIELD_ID}_zscore"] = (filtered_df[SCORE_FIELD_ID] - mean_score) / std_score
print(f"Z-score normalized {SCORE_FIELD_ID}:")
display(filtered_df[[SCORE_FIELD_ID, f"{SCORE_FIELD_ID}_zscore"]].head())

# 3. Group: Mean PHQ-9 score by gender
if GROUP_FIELD_ID in filtered_df.columns:
    group_means = filtered_df.groupby(GROUP_FIELD_ID)[SCORE_FIELD_ID].mean()
    print(f"Mean {SCORE_FIELD_ID} by {GROUP_FIELD_ID}:")
    display(group_means)


## 5. Visualization
Visualize distributions in the primary record set. Example: Distribution of PHQ-9 scores by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if RECORD_SET_ID and SCORE_FIELD_ID and GROUP_FIELD_ID:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=filtered_df, x=SCORE_FIELD_ID, hue=GROUP_FIELD_ID, element='step', stat='density', common_norm=False)
    plt.title(f"Distribution of {SCORE_FIELD_ID} by {GROUP_FIELD_ID}")
    plt.xlabel(SCORE_FIELD_ID)
    plt.ylabel("Density")
    plt.show()

    # Boxplot as well
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=filtered_df, x=GROUP_FIELD_ID, y=SCORE_FIELD_ID)
    plt.title(f"{SCORE_FIELD_ID} by {GROUP_FIELD_ID}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. We examined available record sets and fields, extracted survey data, performed filtering and normalization, and visualized key attributes.

- This workflow demonstrates referencing dataset entities by their `@id`, ensuring robust and reproducible data access.
- Future work might include deeper feature selection, missing data analysis, and model training using fields by their canonical IDs.

For more on Croissant metadata and Python tools: https://mlcommons.org/croissant